# Project II - Chem 277A
### **Genetic analysis for identifying hydrocarbon degrading bacteria**

### Contributors:
Cris Zong, Ethan Chan, Isabella Beatrice Bonomi, Sidney Alexa Brooks

### 1) Objective and Goal of the Project

Objective: 

Goal: 

**Note:** prior to going through this walkthrough, instructions for downloading data will be included in the README markdown file.

In [26]:
# Import standard libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pubchempy as pcp
from Bio import SeqIO

from sklearn.preprocessing import StandardScaler
from statsmodels.api import add_constant, OLS, Logit
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import umap.umap_ as umap
from sklearn.manifold import TSNE
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.datasets import fetch_openml
from sklearn.neighbors import NearestNeighbors

In [ ]:
# Load data
data = pd.read_csv("data.csv", header=0)

X = data.drop(columns=['Compound'])
Y = data['Compound']

# Let's take a look at the data
data.head()

,Mechanism,Compound,Pathway,Subpathway,Protein_ID,Gene,code_mechanism,code_compound,code_subpathway
0,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,P12693,AeAA_alkH_ald,Ae,A,A_
1,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,Q00593,AeAA_alkJ_adh,Ae,A,A_
2,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,Q9WWW2,AeAA_alkJ_adh,Ae,A,A_
3,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,P17051,AeAA_alkS,Ae,A,A_
4,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,Q0VKZ4,AeAA_alkS,Ae,A,A_


In [ ]:
data.Compound.unique() # thats our target

array(['A_Alkanes', 'B_Alkenes', 'C_Aromatics', 'D_Biosurfactants',
       'E_Plastics'], dtype=object)

In [ ]:
# check for missing values
data.isnull().sum()

# encode code_mechanism, code_compound, and code_subathway as categorical codes

data_copy = data.copy() # make a copy of the original data before encoding
data_copy['code_mechanism'] = data['code_mechanism'].astype('category').cat.codes
data_copy['code_compound'] = data['code_compound'].astype('category').cat.codes
data_copy['code_subpathway'] = data['code_subpathway'].astype('category').cat.codes

data_copy.head()

NameError: name 'data' is not defined

In [30]:
seq_dict = {}

for record in SeqIO.parse("HADEG_protein_database_231119.faa", "fasta"):
    protein_id = record.id.split("|")[0]
    seq_dict[protein_id] = str(record.seq)


data["Sequence"] = data["Protein_ID"].map(seq_dict)

data.head()


,Mechanism,Compound,Pathway,Subpathway,Protein_ID,Gene,code_mechanism,code_compound,code_subpathway,Sequence
0,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,P12693,AeAA_alkH_ald,Ae,A,A_,MTIPISLAKLNSSADTHSALEVFNLQKVASSARRGKFGIAERIAAL...
1,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,Q00593,AeAA_alkJ_adh,Ae,A,A_,MYDYIIVGAGSAGCVLANRLSADPSKRVCLLEAGPRDTNPLIHMPL...
2,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,Q9WWW2,AeAA_alkJ_adh,Ae,A,A_,MYDYIIVGAGSAGCVLANRLSADPSKRVCLLEAGPRDTNPLIHMPL...
3,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,P17051,AeAA_alkS,Ae,A,A_,MKIIINNDFPVAKVGADQITTLVSAKVHSCIYRPRLSIADGAAPRV...
4,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,Q0VKZ4,AeAA_alkS,Ae,A,A_,MLCDNLENWCLRMSDRMWARQRFITQGCIERGRLKASAESESKVIL...


In [32]:
data.shape

(629, 10)

In [34]:
len(data.Gene.unique())
len(data.Sequence.unique())

613

In [ ]:
Cat      = "ABCDEFGHIJKLMNOPQRSTUVWXYZ .!?"
bytes_object    = Cat.encode('utf-8') # first, the string has to be turned into a utf-8bit object
Encoding        = {c: f"{ord(c):08b}" for c in Cat}

Encoder  = lambda seq: [Encoding[c] for c in str(seq) if c in Encoding]

data['Sequence'] = data['Sequence'].apply(Encoder)

data.head() 

0      [00100000, 00100000, 00100000, 00100000, 00100...
1      [00100000, 00100000, 00100000, 00100000, 00100...
2      [00100000, 00100000, 00100000, 00100000, 00100...
3      [00100000, 00100000, 00100000, 00100000, 00100...
4      [00100000, 00100000, 00100000, 00100000, 00100...
                             ...                        
624    [00100000, 00100000, 00100000, 00100000, 00100...
625    [00100000, 00100000, 00100000, 00100000, 00100...
626    [00100000, 00100000, 00100000, 00100000, 00100...
627    [00100000, 00100000, 00100000, 00100000, 00100...
628    [00100000, 00100000, 00100000, 00100000, 00100...
Name: Sequence, Length: 629, dtype: object


,Mechanism,Compound,Pathway,Subpathway,Protein_ID,Gene,code_mechanism,code_compound,code_subpathway,Sequence
0,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,P12693,AeAA_alkH_ald,Ae,A,A_,"[00100000, 00100000, 00100000, 00100000, 00100..."
1,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,Q00593,AeAA_alkJ_adh,Ae,A,A_,"[00100000, 00100000, 00100000, 00100000, 00100..."
2,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,Q9WWW2,AeAA_alkJ_adh,Ae,A,A_,"[00100000, 00100000, 00100000, 00100000, 00100..."
3,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,P17051,AeAA_alkS,Ae,A,A_,"[00100000, 00100000, 00100000, 00100000, 00100..."
4,Ae_Aerobic,A_Alkanes,A_Auxiliar_alkane_gene,A_Auxiliar_alkane_gene,Q0VKZ4,AeAA_alkS,Ae,A,A_,"[00100000, 00100000, 00100000, 00100000, 00100..."
